# 🎧 SPEEDWAVE: Audio Speed Converter (Google Colab)

Runs SpeedWave on a free Google Colab machine. **No installation on your own computer.**

**Cara guna / How to use:**
1. Klik **Runtime → Run all** (`Ctrl+F9`). If Colab warns the notebook isn't from Google, click **Run anyway**.
2. Tunggu 2–4 minit / wait 2–4 minutes.
3. Klik butang **🚀 Open SpeedWave** in step 3. The link (`https://….trycloudflare.com`) works in any
   browser, including your phone.
4. If step 3 can't create a link, use step 3B: SpeedWave opens **inside this notebook**.

> The server runs while this Colab tab stays open. Colab stops after about 90 minutes of inactivity, or
> about 12 hours at most, and files are deleted when it stops.
> The step 3 link is public while it runs: anyone who has it can use your SpeedWave (each visitor only sees
> their own files). Don't share it with people you don't trust.

In [ ]:
#@title 1️⃣ Install SpeedWave (FFmpeg + Rubber Band, Python, Node)  { display-mode: "form" }
BRANCH = "claude/audio-speed-converter-ds1og6"  #@param {type:"string"}
REPO = "https://github.com/yuudayo12/AM-Marketplace.git"

import os, re, shutil, subprocess, sys

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r

os.chdir("/content")
print("⬇️  Downloading SpeedWave…")
shutil.rmtree("/content/AM-Marketplace", ignore_errors=True)
sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} /content/AM-Marketplace")
APP = "/content/AM-Marketplace/speedwave"

# FFmpeg: the static build includes librubberband 3.x (high-quality R3 engine).
# If it can't be downloaded, fall back to Ubuntu's ffmpeg (Rubber Band R2, still pitch-preserving).
def ffmpeg_ok(path):
    r = sh(f"{path} -hide_banner -filters", check=False)
    e = sh(f"{path} -hide_banner -encoders", check=False)
    return r.returncode == 0 and " rubberband " in r.stdout and "libmp3lame" in e.stdout and "libvorbis" in e.stdout

FF, FP = "/content/ffmpeg/ffmpeg", "/content/ffmpeg/ffprobe"
if not ffmpeg_ok(FF):
    print("⬇️  Installing FFmpeg (static build with Rubber Band)…")
    os.makedirs("/content/ffmpeg", exist_ok=True)
    sh("curl -fsSL https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz"
       " | tar xJ -C /content/ffmpeg --strip-components=1", check=False)
if not ffmpeg_ok(FF):
    print("⚠️  Static FFmpeg unavailable; installing Ubuntu ffmpeg instead…")
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
    FF, FP = shutil.which("ffmpeg"), shutil.which("ffprobe")
    if not ffmpeg_ok(FF):
        raise SystemExit("❌ No FFmpeg with rubberband/libmp3lame/libvorbis found.")
print("✅", sh(f"{FF} -version").stdout.splitlines()[0])

print("⬇️  Installing Python packages…")
sh(f"{sys.executable} -m pip install -q -r {APP}/backend/requirements.txt")

# Node.js 18+ is needed to build the web interface.
NODE_BIN = ""
r = sh("node --version", check=False)
major = int(re.sub(r"[^0-9.]", "", r.stdout).split(".")[0] or 0) if r.returncode == 0 else 0
if major < 18:
    print("⬇️  Installing Node.js 22…")
    sh("mkdir -p /content/node && curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz"
       " | tar xJ -C /content/node --strip-components=1")
    NODE_BIN = "/content/node/bin"
    os.environ["PATH"] = NODE_BIN + ":" + os.environ["PATH"]
print("🛠️  Building the web interface…")
sh(f"cd {APP}/frontend && npm install --no-audit --no-fund --silent && npm run build --silent")
print("\n✅ Setup complete. Run step 2.")

In [ ]:
#@title 2️⃣ Start the SpeedWave server  { display-mode: "form" }
MAX_UPLOAD_MB = 1000  #@param {type:"integer"}
import json, os, subprocess, sys, time, urllib.request

subprocess.run("pkill -f 'uvicorn app.main:app'", shell=True)
time.sleep(0.5)
env = dict(os.environ,
           SPEEDWAVE_FFMPEG=FF, SPEEDWAVE_FFPROBE=FP,
           SPEEDWAVE_DATA_DIR="/content/speedwave-data",
           SPEEDWAVE_MAX_UPLOAD_MB=str(MAX_UPLOAD_MB),
           SPEEDWAVE_FILE_TTL_MINUTES="240",
           SPEEDWAVE_EMBEDDED="1")   # session cookie that also works inside iframes / over HTTPS tunnels
log = open("/content/speedwave.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000",
                           "--proxy-headers", "--forwarded-allow-ips", "*"],
                          cwd=f"{APP}/backend", env=env, stdout=log, stderr=subprocess.STDOUT,
                          start_new_session=True)  # keeps running if a cell is interrupted
health = None
for _ in range(60):
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2))
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(0.5)
if not health:
    print(open("/content/speedwave.log").read()[-3000:])
    raise SystemExit("❌ The server did not start (log above).")
page = urllib.request.urlopen("http://127.0.0.1:8000/", timeout=5).read()
if b"<div id=\"root\">" not in page:
    raise SystemExit("❌ The web interface was not built. Re-run step 1 and check for errors.")
print("✅ Server running on port 8000")
print("   FFmpeg:     ", health["ffmpeg"])
print("   Rubber Band:", health["rubberband_engine"])

In [ ]:
#@title 3️⃣ 🚀 Open SpeedWave (link for any browser / phone)  { display-mode: "form" }
import os, re, subprocess, time, urllib.request
from IPython.display import HTML, display

if not os.path.exists("/content/cloudflared"):
    subprocess.run("curl -fsSL -o /content/cloudflared "
                   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                   " && chmod +x /content/cloudflared", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
open("/content/cloudflared.log", "w").close()
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(0.5)

ready = False
if PUBLIC_URL:
    print("Waiting for the link to come online…")
    for _ in range(40):  # a new tunnel takes a few seconds to reach Cloudflare's edge
        try:
            urllib.request.urlopen(PUBLIC_URL + "/api/health", timeout=5)
            ready = True
            break
        except Exception:
            time.sleep(1.5)

if PUBLIC_URL:
    note = "" if ready else "<div style='color:#fbbf24;margin-top:10px'>The link may need another 10–30 seconds. If it doesn't open, wait and click again.</div>"
    display(HTML(f"""
<div style="font-family:Inter,system-ui,sans-serif;padding:18px;border-radius:14px;background:#15171d;border:1px solid #2e323d;max-width:640px">
  <div style="font-weight:800;letter-spacing:.2em;color:#c4b5fd;font-size:18px">SPEEDWAVE</div>
  <div style="color:#94a3b8;margin:6px 0 14px">Ready. Opens in a new tab; works on any device.</div>
  <a href="{PUBLIC_URL}" target="_blank" style="display:inline-block;padding:12px 22px;border-radius:12px;color:#fff;font-weight:700;
     text-decoration:none;background:linear-gradient(90deg,#7c3aed,#3b82f6)">🚀 Open SpeedWave</a>
  <div style="color:#64748b;font-size:12px;margin-top:12px;word-break:break-all">{PUBLIC_URL}</div>{note}
</div>"""))
else:
    print("❌ Could not create the link. Use step 3B below. Tunnel log:")
    print(open("/content/cloudflared.log").read()[-1500:])

In [ ]:
#@title 3️⃣B Backup: show SpeedWave inside this notebook  { display-mode: "form" }
#@markdown Use this if step 3 doesn't work. The app appears below this cell.
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=1100)

In [ ]:
#@title 🔧 Troubleshooting: logs / stop  { display-mode: "form" }
ACTION = "show server log"  #@param ["show server log", "show tunnel log", "stop everything"]
import subprocess
if ACTION == "show server log":
    print(open("/content/speedwave.log").read()[-5000:])
elif ACTION == "show tunnel log":
    print(open("/content/cloudflared.log").read()[-5000:])
else:
    subprocess.run("pkill -f 'uvicorn app.main:app'; pkill -f 'cloudflared tunnel'", shell=True)
    print("Stopped.")